# Design an Access Control System

**Company:** MongoDB (GothamLoop question bank) · **Category:** System Design · **Tags:** Onsite Loop, Caching, Concurrency, Databases, Distributed Systems, Security · **Difficulty/Frequency:** Uncommon (3/10)

> **Related:** [`10. LRU_Cache`](../../2.%20Coding_Questions/10.%20LRU_Cache/10.%20LRU_Cache.ipynb) is the cache-invalidation half of this problem; [`9. Broadcast_Message_Bus`](../../2.%20Coding_Questions/9.%20Broadcast_Message_Bus/9.%20Broadcast_Message_Bus.ipynb) is the copy-on-write snapshot swap.

## Concepts

**What this question is really testing:** recognising a **cost asymmetry** and building the whole system around it.

> Role assignments change a few times a day. Authorization checks happen a million times a second.

Everything else follows. When reads outnumber writes by nine orders of magnitude, you do all the work at write time and make reads trivial — even if that means the read path serves data a few seconds stale.

**First-principles primer:**

- **RBAC** is two hops: user → roles → permissions. Nobody grants permissions to people; they grant *roles*, and roles carry permissions. That's what makes "revoke everything this contractor can do" one delete instead of ten thousand.
- **Role inheritance** makes roles a **DAG**. `admin` inherits `editor` inherits `viewer`. Answering "what can `admin` do?" naively means walking that graph on every request.
- **Flattening** walks the DAG *once*, at snapshot build time, and writes down each role's complete effective permission set. The request path then never traverses anything.
- **Deny-precedence.** If any applicable rule says deny, the answer is deny — regardless of how many allows there are. This is not a tiebreak; it is a **security property**. A deny you can override by adding an allow is not a deny.
- **Default-deny.** No matching rule means *no*. Absence of permission is never permission.

**Simple worked example.** Alice holds `editor` and `contractor`:

```
Roles (a DAG):                Flattened at build time:

  viewer ──┐                    viewer      : {(q3.pdf, read): allow}
           ├── editor           editor      : {(q3.pdf, read): allow,
  (base)   │                                   (q3.pdf, write): allow}
           └── contractor       contractor  : {(q3.pdf, write): DENY}

Check: can Alice write q3.pdf?
  editor      says allow
  contractor  says DENY      <- deny wins, immediately
  => DENY
```

Note what did *not* happen at request time: no graph walk, no inheritance resolution. Two hash lookups, one per role.

**The one thing to get right:** the check must be `O(number of roles a user holds)` — about 5 — and **not** proportional to how many permissions those roles carry. That distinction is the difference between a design that works at 1M checks/sec and one that doesn't, and it's where the source answer slips.

## Requirements & Scale

| Functional | Non-functional |
|---|---|
| Users, resources, roles, permissions CRUD | **p99 < 10 ms** per check |
| "Can user X do Y on Z?" | 99.99% availability |
| Inheritable roles (a DAG) | Millions of users; **1M checks/sec** |
| Allow **and** deny, deny wins | Audit logs **tamper-evident** |

**Stated scale:** 5M users, 100k resources, 500 roles per tenant, 1M checks/sec across 10 servers.

In [ ]:
import os, sys, time, random, hashlib
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "capacity.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from capacity import (KB, MB, GB, human_bytes, human_count, human_rate,
                      table, assumption_table)

USERS = 5_000_000
RESOURCES = 100_000
ROLES_PER_TENANT = 500
ACTIONS = 3                     # "a few actions"
BYTES_PER_ENTRY = 20
ROLES_PER_USER = 5
CHECK_SERVERS = 10
CHECKS_PER_SERVER = 100_000
POLL_INTERVAL_SEC = 1.0
REBUILD_SEC = 1.5

assumption_table({
    "Users":                 human_count(USERS),
    "Resources":             human_count(RESOURCES),
    "Roles per tenant":      ROLES_PER_TENANT,
    "Actions per resource":  ACTIONS,
    "Bytes per entry":       f"{BYTES_PER_ENTRY} B",
    "Roles per user":        f"~{ROLES_PER_USER}",
    "Check servers":         CHECK_SERVERS,
    "Checks per server":     human_rate(CHECKS_PER_SERVER, "/s"),
    "Version poll / rebuild": f"{POLL_INTERVAL_SEC}s poll + ~{REBUILD_SEC}s rebuild",
})

## Flattening the inheritance DAG

The one expensive operation, done once per snapshot build rather than once per request. Note that deny-precedence is resolved *here* — the request path never has to reason about it.

In [ ]:
from typing import Dict, List, Tuple, Set

Key = Tuple[str, str]           # (resource, action)


def flatten(role, parents: Dict[str, List[str]],
            own: Dict[str, Dict[Key, str]],
            memo=None, visiting=None) -> Dict[Key, str]:
    """Effective permissions for `role`, inherited included, deny already resolved."""
    memo = {} if memo is None else memo
    visiting = set() if visiting is None else visiting
    if role in memo:
        return memo[role]
    if role in visiting:
        raise ValueError(f"inheritance cycle at {role!r}")
    visiting.add(role)

    effective: Dict[Key, str] = {}
    for parent in parents.get(role, []):                 # inherited first...
        for key, effect in flatten(parent, parents, own, memo, visiting).items():
            if effective.get(key) != "deny":             # ...a deny is never overwritten
                effective[key] = effect
    for key, effect in own.get(role, {}).items():        # ...then the role's own rules
        if effect == "deny" or effective.get(key) != "deny":
            effective[key] = effect

    visiting.discard(role)
    memo[role] = effective
    return effective


parents = {"editor": ["viewer"], "admin": ["editor"], "contractor": []}
own = {
    "viewer":     {("q3.pdf", "read"): "allow"},
    "editor":     {("q3.pdf", "write"): "allow"},
    "admin":      {("q3.pdf", "delete"): "allow"},
    "contractor": {("q3.pdf", "write"): "deny"},
}

memo = {}
for r in ("viewer", "editor", "admin", "contractor"):
    flat = flatten(r, parents, own, memo)
    table([(f"{k[0]} / {k[1]}", v) for k, v in sorted(flat.items())],
          title=f"FLATTENED: {r}")

assert flatten("admin", parents, own, memo) == {
    ("q3.pdf", "read"): "allow",
    ("q3.pdf", "write"): "allow",
    ("q3.pdf", "delete"): "allow",
}, "admin inherits transitively through editor -> viewer"

# A deny in a PARENT must survive an allow in the child.
p2 = {"child": ["locked"]}
o2 = {"locked": {("x", "read"): "deny"}, "child": {("x", "read"): "allow"}}
assert flatten("child", p2, o2, {})[("x", "read")] == "deny", \
    "deny must not be overridable by an inherited-from child's allow"
print("\n  Deny survives an overriding allow in the child role.")

# Cycles must be rejected at write time, not discovered at rebuild time.
try:
    flatten("a", {"a": ["b"], "b": ["a"]}, {}, {})
    raise SystemExit("should have raised")
except ValueError as e:
    print(f"  Cycle detected: {e}")

### ⚠️ Defect 1 — the schema cannot express deny-precedence at all

```sql
UNIQUE (tenant_id, resource_id, action)
```

A `permissions` row is `(resource, action, effect)`. That constraint permits **one row per `(tenant, resource, action)`** — so a resource+action is *either* an allow *or* a deny, never both.

Yet the answer's own talking point describes exactly the forbidden case:

> *"when a user has one role that allows read on a file and another that denies it, the deny must win"*

Two roles, same `(file, read)`, opposite effects. Storing that needs two rows, and the second `INSERT` fails.

In [ ]:
class UniqueViolation(Exception):
    pass


class PermissionsTable:
    """The schema as written, with the stated UNIQUE constraint enforced."""

    def __init__(self, key_includes_effect: bool):
        self.rows: Dict[tuple, dict] = {}
        self.key_includes_effect = key_includes_effect

    def insert(self, tenant, resource, action, effect):
        key = ((tenant, resource, action, effect) if self.key_includes_effect
               else (tenant, resource, action))
        if key in self.rows:
            raise UniqueViolation(f"duplicate key {key}")
        self.rows[key] = {"resource": resource, "action": action, "effect": effect}
        return key


# --- As specified: UNIQUE (tenant_id, resource_id, action) ---
t = PermissionsTable(key_includes_effect=False)
t.insert("acme", "q3.pdf", "read", "allow")          # the editor role's rule
try:
    t.insert("acme", "q3.pdf", "read", "deny")       # the contractor role's rule
    raise SystemExit("should have raised")
except UniqueViolation as e:
    print(f"  AS SPECIFIED: {e}")
    print("  -> the deny rule cannot be stored. Deny-precedence is unreachable.")

# --- Corrected: UNIQUE (tenant_id, resource_id, action, effect) ---
t2 = PermissionsTable(key_includes_effect=True)
t2.insert("acme", "q3.pdf", "read", "allow")
t2.insert("acme", "q3.pdf", "read", "deny")          # now permitted
assert len(t2.rows) == 2
print("\n  CORRECTED:    both rows stored; the flattening step can now do its job.")

print("\n  => Note the failure mode: not a slow query, but a REQUIREMENT that")
print("     silently never fires. Nobody notices until an audit asks why a deny")
print("     rule was never applied.")

### ⚠️ Defect 2 — "union the effective permission sets" is not O(1)

The check flow says to *"union the effective permission sets of those roles"*, then claims *"O(1) per check — hash map lookups only"*. Those cannot both be true. A union is **O(total size of the sets)**.

A user with 5 roles of 10,000 permissions each does 50,000 operations and allocates a 50,000-entry map — to answer a question about **one** key.

You never need the union. Probe each role's map for the single key in question: **O(roles per user)**, independent of permission count.

In [ ]:
def check_union(snapshot_user_roles, snapshot_role_perms, user, key):
    """As written in the answer: build the union, then look up."""
    effective: Dict[Key, str] = {}
    for role in snapshot_user_roles[user]:
        for k, effect in snapshot_role_perms[role].items():
            if effect == "deny" or effective.get(k) != "deny":
                effective[k] = effect
    return effective.get(key, "deny")            # default-deny


def check_probe(snapshot_user_roles, snapshot_role_perms, user, key):
    """The fix: probe each role for the ONE key, short-circuit on deny."""
    decision = "deny"                            # default-deny
    for role in snapshot_user_roles[user]:
        effect = snapshot_role_perms[role].get(key)
        if effect == "deny":
            return "deny"                        # deny wins; stop immediately
        if effect == "allow":
            decision = "allow"
    return decision


def make_snapshot(perms_per_role, n_roles=ROLES_PER_USER, seed=0):
    rnd = random.Random(seed)
    role_perms, user_roles = {}, {"alice": [f"r{i}" for i in range(n_roles)]}
    for i in range(n_roles):
        role_perms[f"r{i}"] = {(f"res{rnd.randrange(1_000_000)}", "read"): "allow"
                               for _ in range(perms_per_role)}
    role_perms["r0"][("q3.pdf", "read")] = "allow"
    return user_roles, role_perms


# Both must agree on the answer - the fix has to be a pure optimisation.
for pr in (10, 100, 1_000):
    ur, rp = make_snapshot(pr)
    for key in (("q3.pdf", "read"), ("nonexistent", "read")):
        assert check_union(ur, rp, "alice", key) == check_probe(ur, rp, "alice", key)
    rp["r3"][("q3.pdf", "read")] = "deny"            # a deny in another role
    assert check_union(ur, rp, "alice", ("q3.pdf", "read")) == "deny"
    assert check_probe(ur, rp, "alice", ("q3.pdf", "read")) == "deny"
print("  Both approaches agree on every case, including deny-precedence.\n")

# ...but they do not cost the same.
rows, N = [], 20_000
prev = None
for pr in (10, 100, 1_000, 10_000):
    ur, rp = make_snapshot(pr)
    key = ("q3.pdf", "read")
    reps = max(1, N // pr)

    t0 = time.perf_counter()
    for _ in range(reps):
        check_union(ur, rp, "alice", key)
    t_union = (time.perf_counter() - t0) / reps * 1e6

    t0 = time.perf_counter()
    for _ in range(reps * 50):
        check_probe(ur, rp, "alice", key)
    t_probe = (time.perf_counter() - t0) / (reps * 50) * 1e6

    growth = f"{t_union / prev:.1f}x" if prev else "-"
    prev = t_union
    rows.append((f"{pr:,} perms/role", f"union {t_union:9.1f} us  ({growth:>5} growth)"
                                       f"   probe {t_probe:6.2f} us"))
table(rows, title="COST PER CHECK (5 roles per user)")

print("\n  => union grows ~10x per 10x more permissions; probe is flat.")
print("     The probe answers the same question without ever materialising")
print("     a set it is going to throw away.")

In [ ]:
# What the difference means at the stated 1M checks/sec.
PERMS_PER_ROLE = 10_000
union_ops = ROLES_PER_USER * PERMS_PER_ROLE
probe_ops = ROLES_PER_USER
target_qps = CHECK_SERVERS * CHECKS_PER_SERVER

table([
    ("Ops per check, union",  f"{union_ops:,}"),
    ("Ops per check, probe",  f"{probe_ops:,}"),
    ("Reduction",             f"{union_ops // probe_ops:,}x"),
    ("", ""),
    ("Target throughput",     human_rate(target_qps, " checks/s")),
    ("Fleet ops/s, union",    human_count(union_ops * target_qps) + " ops/s"),
    ("Fleet ops/s, probe",    human_count(probe_ops * target_qps) + " ops/s"),
], title="AT 1M CHECKS/SEC")

assert target_qps == 1_000_000, "the answer's stated 1M checks/sec"
assert union_ops * target_qps == 50_000_000_000
print("\n  => 50 BILLION operations per second is not 'a handful of hash lookups'.")
print("     5 million is.")

### ⚠️ Defect 3 — the memory estimate sizes one tenant; the architecture stores all of them

The scaling math uses *"500 roles **per tenant**"* to reach 150M entries ≈ 3 GB. But the check-flow section says each server holds *"the entire permission graph **for all tenants**"*.

So the real footprint is 3 GB **× tenant count**. Two further omissions compound it: user-role assignments are never counted, and 20 bytes/entry assumes a hash table with zero slack.

In [ ]:
entries = ROLES_PER_TENANT * RESOURCES * ACTIONS
snapshot_per_tenant = entries * BYTES_PER_ENTRY

# What the 3 GB figure leaves out.
user_role_bytes = USERS * ROLES_PER_USER * 16          # uuid pair, interned
LOAD_FACTOR = 0.7                                      # real open-addressed maps
realistic = snapshot_per_tenant / LOAD_FACTOR

table([
    ("Flattened entries (1 tenant)", human_count(entries)),
    ("At 20 B/entry",               human_bytes(snapshot_per_tenant)),
    ('  ...the answer says',        '"under 3 GB"'),
    ("", ""),
    ("+ user-role assignments",     human_bytes(user_role_bytes)),
    ("+ hash slack (70% load)",     human_bytes(realistic - snapshot_per_tenant)),
    ("Realistic, ONE tenant",       human_bytes(realistic + user_role_bytes)),
], title="SNAPSHOT MEMORY")

assert entries == 150_000_000, "the answer's stated ~150M entries"
assert snapshot_per_tenant == 3 * GB, "exactly 3 GB - not 'under' 3 GB"

SERVER_RAM = 16 * GB
per_tenant_real = realistic + user_role_bytes
rows = []
for tenants in (1, 2, 5, 10, 100):
    total = per_tenant_real * tenants
    fits = "fits" if total < SERVER_RAM * 0.8 else "DOES NOT FIT"
    rows.append((f"{tenants:>3} tenants", f"{human_bytes(total):>10}   {fits}"))
table(rows, title="'ALL TENANTS' ON A 16 GB SERVER")

max_tenants = int(SERVER_RAM * 0.8 / per_tenant_real)
assert max_tenants < 4, "a 16 GB server holds only a few tenants at this size"
print(f"\n  => A 16 GB server holds {max_tenants} tenants, not 'all of them'.")
print("     The fix the design already sets up: shard SERVERS by tenant. Per-tenant")
print("     versioning means a server only rebuilds for the tenants it owns - so")
print("     tenant sharding improves memory AND rebuild frequency at once.")

## Invalidation: the staleness window is a security window

The snapshot is a cache, and this cache decides who may do what. A version counter plus polling gives a bounded staleness — but "bounded" is doing real work in that sentence, because during the window a **revoked** user still has access.

In [ ]:
worst_case = POLL_INTERVAL_SEC + REBUILD_SEC

table([
    ("Poll interval",              f"{POLL_INTERVAL_SEC:.2f} s"),
    ("Rebuild time",               f"{REBUILD_SEC:.2f} s"),
    ("Worst-case staleness",       f"{worst_case:.2f} s"),
    ("", ""),
    ("Checks served stale, 1 server",
     human_count(CHECKS_PER_SERVER * worst_case)),
    ("Checks served stale, fleet",
     human_count(CHECK_SERVERS * CHECKS_PER_SERVER * worst_case)),
], title="STALENESS AFTER A REVOCATION")

assert 2 <= worst_case <= 3, "the answer's stated 2-3 seconds"
print("\n  => 2.5 million checks may use pre-revocation permissions. For 'user")
print("     removed from the editors group' that is fine. For 'employee")
print("     terminated, walked out by security' it is not.")
print("\n     So: grant changes may propagate lazily; REVOCATIONS should not.")
print("     Push them synchronously (pub/sub, ack before the API returns), or")
print("     keep a small deny-list checked on every request and consulted ahead")
print("     of the snapshot. The deny-list is tiny and short-lived - entries can")
print("     be dropped once the snapshot version passes the revocation version.")

# The deny-list closes the window without touching the fast path's shape.
revoked_until_version = {("alice", "q3.pdf", "write"): 42}

def check_with_denylist(user, key, snapshot_version, base_decision):
    v = revoked_until_version.get((user,) + key)
    if v is not None and snapshot_version < v:
        return "deny"                       # revocation not yet in this snapshot
    return base_decision

assert check_with_denylist("alice", ("q3.pdf", "write"), 41, "allow") == "deny"
assert check_with_denylist("alice", ("q3.pdf", "write"), 42, "allow") == "allow"
print("\n  Stale snapshot (v41): denied. Caught-up snapshot (v42): the deny-list")
print("  entry is inert and can be garbage-collected.")

## Audit: what a hash chain does and does not give you

`hash_i = SHA-256(hash_{i-1} || payload_i)` makes the log **tamper-evident**: changing any historical row changes its hash, which changes every hash after it.

It is *not* tamper-**proof**. Anyone who can write to the table can recompute the whole chain forward and leave it perfectly consistent. What stops that is the **anchor** — periodically publishing the current head hash somewhere the attacker cannot rewrite. Verification then only trusts the segment from the last anchor onward.

In [ ]:
GENESIS = "0" * 64

def chain_hash(prev: str, payload: str) -> str:
    return hashlib.sha256((prev + payload).encode()).hexdigest()

def build_log(payloads):
    log, prev = [], GENESIS
    for p in payloads:
        prev = chain_hash(prev, p)
        log.append({"payload": p, "hash": prev})
    return log

def verify(log, anchor_hash=GENESIS, anchor_index=0):
    prev = anchor_hash
    for i in range(anchor_index, len(log)):
        prev = chain_hash(prev, log[i]["payload"])
        if prev != log[i]["hash"]:
            return False, i
    return True, None


events = [f"user{i} read file{i} -> allow" for i in range(6)]
log = build_log(events)
assert verify(log)[0]
print(f"  Clean log of {len(log)} events verifies.")

# Tamper with row 2 and leave the hashes alone: detected immediately.
tampered = [dict(r) for r in log]
tampered[2]["payload"] = "user2 read file2 -> DENY (falsified)"
ok, at = verify(tampered)
assert not ok and at == 2
print(f"  Edited row {at}, hashes untouched  -> detected at row {at}.")

# Tamper AND recompute the chain forward: undetectable without an anchor.
recomputed = build_log([r["payload"] for r in tampered])
assert verify(recomputed)[0], "a fully recomputed chain is internally consistent"
print("  Edited row 2 and recomputed the chain -> verifies clean. NOT detected.")

# The anchor is what closes that hole.
anchor_index, anchor_hash = 4, log[3]["hash"]     # published externally at row 4
ok, at = verify(recomputed, anchor_hash, anchor_index)
assert not ok, "the recomputed chain no longer matches the external anchor"
print(f"  ...but against an anchor published at row {anchor_index}: DETECTED.")

print("\n  => A hash chain alone proves only INTERNAL consistency. The anchor is")
print("     what makes it evidence, and it is only as good as the write-once")
print("     store you put it in. Say where the anchor lives, or the property")
print("     you are claiming does not exist.")

## Discussion — the follow-ups

- **Direct user permissions, bypassing roles.** Add a `user_permissions` map to the snapshot and probe it alongside the role maps — one more `O(1)` lookup, and it participates in deny-precedence like any other source. Resist making it a *fourth* thing the check has to union.
- **A user deactivated mid-session.** Two separate problems. Put `status` in the snapshot and check it first, and you fix the *authorization* path — but not sessions already issued. If a bearer token is valid for an hour, deactivation does nothing for an hour unless the check is consulted per request, which is exactly the argument for making authz a call rather than a claim baked into a token. Combine with the deny-list above for immediate effect.
- **Hierarchical resources** (`/reports` grants imply `/reports/q3.pdf`). Longest-prefix matching is the right semantic, but it breaks the flat hash lookup: you now walk from the resource up to the root, probing each ancestor — `O(path depth)`, still tiny. Precomputing every descendant instead is a trap: one grant on `/` would expand to every resource in the tenant. This is the same fan-out trade as [`14. Path_Resolving`](../../2.%20Coding_Questions/14.%20Path_Resolving/14.%20Path_Resolving.ipynb).
- **OPA or Cedar instead.** Genuinely a reasonable choice, and the honest comparison is: policy engines give you expressive, versionable policy-as-code and attribute-based rules that RBAC can't express (`owner == user.id`, time-of-day, IP range); a purpose-built flattened snapshot gives you microsecond checks and full control over the memory layout. Most teams should take the engine. Say *which* constraint would push you to build.
- **The same user in multiple tenants.** Already handled: `user_roles` reaches tenant scope through `roles.tenant_id`, and inheritance never crosses tenants — so a user simply holds disjoint role sets per tenant. Worth stating that the *check* must be scoped by the `tenant_id` in the request and never infer it from the user, or you've built a cross-tenant escalation.

## Patterns learned

- **Find the cost asymmetry and build around it.** Writes are rare, reads are constant, so pay at write time. That single observation produces the flattened snapshot, the version counter, and the staleness budget.
- **Precompute the traversal, not the answer.** Flattening the DAG removes graph walking from the request path while keeping the snapshot proportional to roles, not to users.
- **Resolve deny-precedence at build time.** The request path should never reason about conflicting rules; it should read an already-resolved verdict.
- **Don't materialise what you're about to discard.** "Union the sets then look up one key" is the archetypal version of this bug — 50,000 operations to answer a question about one. Probe instead.
- **Check whether your schema can express your requirements.** A `UNIQUE` constraint here made a stated security feature *unrepresentable*. It fails closed and silently, which is the worst way to fail.
- **Say which unit your estimate is in.** "3 GB" per tenant and "all tenants in memory" are individually fine and jointly impossible.
- **Cache staleness on an authorization path is a security window.** Grants can propagate lazily; revocations should not. A short-lived deny-list closes the gap without changing the fast path.
- **Default-deny, always.** Absence of a rule is never permission.
- **Tamper-*evident* is not tamper-*proof*.** A hash chain proves internal consistency; only an external anchor makes it evidence. Name where the anchor lives.
- **Availability of authorization beats completeness of audit.** Never fail a check because the audit write timed out — but do log the gap.